## Baseline and Evaluation Protocol

In this part we set up the evaluation protocol shared by every model, written in `helpers/utils.py`, and evaluate the baseline: a model that always predicts the mean of the train folds. Every model of the next notebooks must beat it.

### Protocol (`helpers/utils.py`)

- **Rows:** One regression per target, trained on the rows where this target is known (`get_xy`). `charpy_temp_C` is an input of the Charpy model only.
- **Inputs:** The 30 inputs of the EDA notebook (chemistry, welding parameters, PWHT and welding process), standardised inside a `Pipeline` (`build_pipeline`: `StandardScaler` + model), so that the scaler is fitted on the train folds only.
- **Folds:** 5 folds grouped on `input_group` and stratified on the alloy family (`StratifiedGroupKFold`, seed 42). `load_train` gives each deposit its fold once for all, so the folds are the same for every target and every person, and every fold holds Cr-Mo welds.
- **Regression metrics:** MAE and RMSE of the out-of-fold predictions, with their standard deviation over the 5 folds. The MAE is in the unit of the target and easy to read, the RMSE penalises the large errors more.
- **Conformity:** The thresholds of `definition_bonne_soudure.md` are applied to the out-of-fold predictions, with the thresholds of the family of each row. For Charpy, the model predicts the energy at the reference temperature of the family (-40 °C for C-Mn, +20 °C for Cr-Mo), by changing `charpy_temp_C` in its input.
- **Classification metrics:** F1 and recall, with "non-conforming" as the positive class, and accuracy. They are computed per criterion on the rows where it is decided, and for the full label on the labelled weld deposits. The recall is the share of bad welds the model catches, which is what matters most to a manufacturer. The F1 also penalises the false alarms. The accuracy is the share of welds classified right: it is easy to read, but a model that declares every weld conforming already gets a high accuracy, so it only complements the F1 and the recall.
- **Results table:** `results.csv`, one row per model, target and family (`all`, `C-Mn`, `CrMo2`, `CrMo91`). `save_results` replaces the previous rows of the same model.

### How to evaluate a model

The model notebooks go in this folder, next to the `helpers/` folder, so that `from helpers.utils import ...` works. The hyperparameters are tuned per target by a grid search on the same folds, then `evaluate` cross-validates the best model of each target:

```python
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from helpers.utils import SEED, TARGETS, load_train, get_xy, build_pipeline, evaluate, save_results

train = load_train()

best = {}
for target in TARGETS:
    X, y, cv = get_xy(train, target)
    search = GridSearchCV(
        build_pipeline(RandomForestRegressor(random_state=SEED)),
        {"model__max_depth": [5, 10, None]},
        cv=cv,
        scoring="neg_mean_absolute_error",
    )
    search.fit(X, y)
    best[target] = search.best_estimator_

save_results(evaluate("Random forest", best, train))
```

### Operations in order

1. Load the train set with `load_train`.
2. Check the folds: each deposit in a single fold, deposits per fold and family, rows per fold and target.
3. Count the decided and non-conforming rows of each criterion.
4. Count the labelled deposits, and compute the F1 of a rule that declares every weld non-conforming.
5. Evaluate the baseline, a `DummyRegressor` that predicts the mean of the train folds.
6. Save it to `results.csv`.

In [1]:
import pandas as pd
from sklearn.dummy import DummyRegressor

from helpers.utils import TARGETS, load_train, get_xy, true_criteria, deposit_labels, evaluate, save_results

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_rows", None)

In [3]:
train = load_train()

print(train.shape)
print("Deposits in more than one fold:", (train.groupby("input_group")["fold"].nunique() > 1).sum())

deposits = train.drop_duplicates("input_group")
pd.crosstab(deposits["family"], deposits["fold"], margins=True)

(1332, 40)
Deposits in more than one fold: 0


fold,0,1,2,3,4,All
family,,,,,,
C-Mn,84,83,84,84,85,420
CrMo2,6,7,7,9,6,35
CrMo91,9,9,8,7,8,41
All,99,99,99,100,99,496


- **fold:** Each deposit is in a single fold, and each fold holds 99 or 100 deposits, of which 6 to 9 CrMo2 and 7 to 9 CrMo91. The train set has no CrMo9 deposit.

In [4]:
pd.DataFrame({target: train.loc[get_xy(train, target)[0].index, "fold"].value_counts().sort_index() for target in TARGETS})

,yield_strength_MPa,uts_MPa,elongation_pct,charpy_toughness_J
fold,,,,
0,132,125,108,137
1,124,121,115,145
2,122,115,110,147
3,124,117,113,143
4,118,118,112,151


The folds are fixed on the deposits, not on each target, so the number of rows per fold varies a little from one target to another (from 108 to 151 rows).

In [5]:
criteria = true_criteria(train)

pd.DataFrame({
    "decided": criteria.notna().sum(),
    "non_conforming": (criteria == 0).sum(),
    "non_conforming_pct": ((criteria == 0).sum() / criteria.notna().sum() * 100).round(1),
})

,decided,non_conforming,non_conforming_pct
yield_strength_MPa,619,102,16.5
uts_MPa,595,198,33.3
elongation_pct,557,48,8.6
charpy_toughness_J,315,29,9.2


- **yield_strength_MPa, uts_MPa, elongation_pct:** Decided on every row where they are measured, except the Cr-Mo row without PWHT, which is outside the conditions of its standard. The UTS fails most often (33% of the rows), on C-Mn welds outside the 500-640 MPa range.
- **charpy_toughness_J:** Only 315 of the 723 Charpy rows are decided. The others were tested at a temperature that does not allow a conclusion (see `definition_bonne_soudure.md`).

In [6]:
labels = deposit_labels(train, criteria, criteria)

summary = labels.groupby("family")["true_nc"].agg(["size", "sum"])
summary.loc["all"] = [len(labels), labels["true_nc"].sum()]
summary.columns = ["deposits", "non_conforming"]
display(summary)

p = labels["true_nc"].mean()
print("Share of non-conforming deposits:", round(p, 3))
print("F1 of 'every weld is non-conforming':", round(2 * p / (1 + p), 3))

,deposits,non_conforming
family,,
C-Mn,161,79
CrMo2,8,1
CrMo91,6,3
all,175,83


Share of non-conforming deposits: 0.474
F1 of 'every weld is non-conforming': 0.643


`deposit_labels` takes the measured criteria twice here, as the true and as the predicted criteria: only the true label `true_nc` is used.

- **Labelled deposits:** 175, of which 83 are non-conforming. Only 14 are Cr-Mo (8 CrMo2 and 6 CrMo91), so the label metrics of the Cr-Mo families are only indicative.
- **Reference F1:** A rule that declares every weld non-conforming has a recall of 1 and an F1 of 2p / (1 + p), with p the share of non-conforming deposits, so 0.64 here. On the label, a model is only useful if its F1 is above this value.

In [7]:
baseline = evaluate("Baseline (mean)", DummyRegressor(strategy="mean"), train)
baseline.round(2)

,model,target,family,n,MAE,RMSE,MAE_std,RMSE_std,n_clf,n_nc,F1_nc,recall_nc,accuracy,precision_good
0,Baseline (mean),yield_strength_MPa,all,620,72.57,94.41,6.91,8.38,619,102,0.0,0.0,0.84,0.84
1,Baseline (mean),yield_strength_MPa,C-Mn,552,68.25,85.67,7.31,5.59,552,102,0.0,0.0,0.82,0.82
2,Baseline (mean),yield_strength_MPa,CrMo2,30,65.15,94.58,43.92,50.27,29,0,0.0,0.0,1.00,1.00
3,Baseline (mean),yield_strength_MPa,CrMo91,38,141.25,178.20,50.25,52.52,38,0,0.0,0.0,1.00,1.00
4,Baseline (mean),uts_MPa,all,596,73.45,92.19,6.59,10.66,595,198,0.0,0.0,0.67,0.67
5,Baseline (mean),uts_MPa,C-Mn,531,68.13,83.13,6.94,6.15,531,198,0.0,0.0,0.63,0.63
6,Baseline (mean),uts_MPa,CrMo2,34,80.85,111.53,47.35,53.61,33,0,0.0,0.0,1.00,1.00
7,Baseline (mean),uts_MPa,CrMo91,31,156.29,177.20,25.79,42.60,31,0,0.0,0.0,1.00,1.00
8,Baseline (mean),elongation_pct,all,558,4.01,4.85,0.25,0.20,557,48,0.0,0.0,0.91,0.91
9,Baseline (mean),elongation_pct,C-Mn,497,3.76,4.55,0.30,0.24,497,37,0.0,0.0,0.93,0.93


- **MAE:** 73 MPa on the yield strength, 73 MPa on the UTS, 4.0% on the elongation and 40 J on the Charpy energy: the error of a model that knows nothing about the weld. It is about twice as large on the CrMo91 welds (141 and 156 MPa on the yield strength and the UTS), which are far from the mean of the train set.
- **F1, recall:** 0 everywhere. The mean of each target passes the thresholds, so the baseline declares every weld conforming and does not catch a single bad weld.
- **accuracy:** Still 0.53 on the label (92 conforming deposits out of 175), and from 0.67 (UTS) to 0.91 (elongation and Charpy) on the criteria, the share of conforming rows. This is why the models are compared on the F1 and the recall, the accuracy is only a complement.

In [8]:
results = save_results(baseline)
results[results["family"] == "all"].round(2)

,model,target,family,n,MAE,RMSE,MAE_std,RMSE_std,n_clf,n_nc,F1_nc,recall_nc,accuracy,precision_good
0,AdaBoost Tuned,yield_strength_MPa,all,620,43.79,57.89,3.28,6.49,619,102,0.02,0.01,NaN,NaN
4,AdaBoost Tuned,uts_MPa,all,596,40.21,51.50,3.04,6.88,595,198,0.58,0.42,NaN,NaN
8,AdaBoost Tuned,elongation_pct,all,558,2.41,3.03,0.13,0.13,557,48,0.18,0.10,NaN,NaN
12,AdaBoost Tuned,charpy_toughness_J,all,723,27.37,35.74,3.29,4.53,315,29,0.00,0.00,NaN,NaN
16,AdaBoost Tuned,label,all,<NA>,NaN,NaN,NaN,NaN,175,83,0.31,0.18,NaN,NaN
20,Bagging,yield_strength_MPa,all,620,35.24,49.40,3.77,6.82,619,102,0.68,0.57,0.91,0.92
24,Bagging,uts_MPa,all,596,30.86,44.45,3.52,8.47,595,198,0.67,0.56,0.82,0.81
28,Bagging,elongation_pct,all,558,1.95,2.51,0.15,0.23,557,48,0.62,0.50,0.95,0.95
32,Bagging,charpy_toughness_J,all,723,19.11,28.16,2.85,4.36,315,29,0.17,0.10,0.91,0.92
36,Bagging,label,all,<NA>,NaN,NaN,NaN,NaN,175,83,0.81,0.72,0.84,0.79


`results.csv` now holds the baseline. Each model notebook adds its rows with `save_results`, and the models are compared with `load_results()`, for example `load_results().pivot_table(index=["target", "family"], columns="model", values="MAE", sort=False)`.